# TP2 sur Colab : entraîner, évaluer et sauvegarder un modèle

Variante du TP2 pour les postes sans installation. Mêmes étapes, mêmes contrôles.

**Avant de commencer :** clonez votre dépôt du TP1, puis téléversez le kit TP2 (`tp2-semaine2.zip`)
avec l'icône dossier du panneau de gauche.


## Étape A. Récupérer le dépôt et installer le kit (contrôle C1)


In [1]:
ORGANISATION = "anasmouquinee"   # votre compte ou organisation
DEPOT = "sentiment-app"

!git clone https://github.com/{ORGANISATION}/{DEPOT}.git
!unzip -q -o tp2-semaine2.zip
!cd tp2-semaine2 && python appliquer_tp2.py ../{DEPOT}
%cd {DEPOT}

Cloning into 'sentiment-app'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 45 (delta 10), reused 40 (delta 5), pack-reused 0
Receiving objects: 100% (45/45), 35.20 KiB | 2.51 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Dépôt cible : /content/sentiment-app
  ajoute         src/make_demo.py
  ajoute         src/train.py
  ajoute         src/evaluate.py
  ajoute         src/predict.py
  ajoute         tests/test_model.py
  ajoute         notebooks/TP2-colab.ipynb
  ajoute         TP2-ENONCE.md
  ajoute         MARCHE-A-SUIVRE-TP2.md
  ajoute         .gitignore : !models/metrics.json

Kit installé. Suite : lire MARCHE-A-SUIVRE-TP2.md, puis python src/make_demo.py
/content/sentiment-app


In [2]:
!pip install -q pytest pyarrow
!python -m pytest -q --deselect tests/test_environment.py::test_virtualenv_active
# attendu : 2 failed, 37 passed, 2 skipped (test_virtualenv_active écarté sur Colab)


.....................................FFss                                [100%]
=================================== FAILURES ===================================
__________________________ test_metriques_completes ___________________________
tests/test_model.py:36: in test_metriques_completes
    metriques = lire_metriques()
tests/test_model.py:23: in lire_metriques
    assert METRIQUES.exists(), "models/metrics.json absent : lancez python src/train.py"
E   AssertionError: models/metrics.json absent : lancez python src/train.py
E   assert False
_________________________ test_f1_au_dessus_du_seuil __________________________
tests/test_model.py:43: in test_f1_au_dessus_du_seuil
    metriques = lire_metriques()
tests/test_model.py:23: in lire_metriques
    assert METRIQUES.exists(), "models/metrics.json absent : lancez python src/train.py"
E   AssertionError: models/metrics.json absent : lancez python src/train.py
E   assert False
=========================== short test summary info =======

## Étape B. Générer le jeu de démonstration (contrôle C2)


In [3]:
!python src/make_demo.py
import pandas as pd
df = pd.read_csv('data/demo_reviews.csv')
print(df.label.value_counts())
df.sample(5, random_state=0)


290 avis écrits dans data/demo_reviews.csv
  145 positifs, 145 négatifs
  dont 100 avis mitigés et 30 neutres, volontairement difficiles
label
positive    145
negative    145
Name: count, dtype: int64


,review,label
55,"Produit conforme à la description, très satisfait.",positive
124,Se casse au bout de deux utilisations.,negative
181,"Taille trop petite et retour payant au déballage, jusqu'à ce que solide, léger, exactement ce que je cherchais.",positive
210,"Aucune réponse du service client, vraiment.",negative
271,"Conforme à la fiche produit, se casse au bout de deux utilisations.",negative


Repérez un avis mitigé : une première moitié d'un sentiment, un retournement, une seconde moitié de l'autre.
L'étiquette suit la seconde moitié. Cette structure explique toutes les erreurs de l'étape D.


## Étape C. Entraîner et sauvegarder (contrôle C3)


In [4]:
!python src/train.py
# attendu : F1 macro 0.879 sur le jeu de démonstration


Source : demo, 290 avis (145 négatifs)
232 avis d'entraînement, 58 de test
Vocabulaire : 316 termes
F1 macro sur le jeu de test : 0.879 (seuil 0.85)
Modèle : models/sentiment.joblib
Métriques : models/metrics.json (versionné dans Git, contrairement au modèle)


In [5]:
!cat models/metrics.json


{
  "source": "demo",
  "date": "2026-10-12",
  "n_train": 232,
  "n_test": 58,
  "f1_macro": 0.879,
  "accuracy": 0.879,
  "recall_negative": 0.931,
  "seuil_f1": 0.85
}


## Étape D. Évaluer et lire les erreurs (contrôle C4)


In [6]:
!python src/evaluate.py
# attendu : 7 erreurs sur 58 avis, toutes sur des avis mitigés


Évaluation sur 58 avis (demo)

              precision    recall  f1-score   support

    negative      0.844     0.931     0.885        29
    positive      0.923     0.828     0.873        29

    accuracy                          0.879        58
   macro avg      0.883     0.879     0.879        58
weighted avg      0.883     0.879     0.879        58

Matrice de confusion (lignes = vérité, colonnes = prédiction)
            negative  positive
  negative        27         2
  positive         5        24

2 négatifs classés positifs, 5 positifs classés négatifs

7 erreurs au total. Trois exemples :
  vérité=positive  prédit=negative  | Taille trop petite et retour payant au déballage, sauf que ensuite produit conforme à la d
  vérité=positive  prédit=negative  | Commande jamais reçue et remboursement en attente au début, sauf que ensuite je recommande
  vérité=negative  prédit=positive  | Service impeccable et livraison en avance sur le papier, sauf que ensuite produit cassé à 


Écrivez trois phrases dans `docs/semaine2.md` : quelles erreurs, pourquoi, et ce qu'il faudrait pour les corriger.


## Étape E. Le vrai jeu, les tests, le push (contrôle C5)


In [7]:
!python src/train.py --allocine
# 20 000 avis publics, attendu : F1 macro 0.920


Source : allocine, 20000 avis (10204 négatifs)
16000 avis d'entraînement, 4000 de test
Vocabulaire : 149529 termes
F1 macro sur le jeu de test : 0.920 (seuil 0.85)
Modèle : models/sentiment.joblib
Métriques : models/metrics.json (versionné dans Git, contrairement au modèle)


In [8]:
!python -m pytest -q --deselect tests/test_environment.py::test_virtualenv_active
# attendu : 41 passed (les 42 du TP moins test_virtualenv_active)


.........................................                                [100%]
41 passed, 1 deselected in 1.48s


In [9]:
from getpass import getpass
JETON = getpass("Jeton d'accès personnel GitHub : ")

!git config user.name "Assia Nouaoury" && git config user.email "assianouaoury@gmail.com"
!git add -A && git commit -q -m "Semaine 2 : modele de sentiment, metriques et tests"
!git remote set-url origin https://{JETON}@github.com/{ORGANISATION}/{DEPOT}.git
!git push

Jeton d'accès personnel GitHub : ········
To https://github.com/anasmouquinee/sentiment-app.git
   ceaa7a5..22193aa  main -> main


## Les deux différences avec un poste local

**Pas d'environnement virtuel :** `test_virtualenv_active` est écarté, comme au TP1.

**Rien ne survit à la session :** poussez sur GitHub avant de fermer Colab, sinon le travail est perdu.
Le modèle `models/sentiment.joblib`, lui, n'a pas à être conservé : il se réentraîne en une commande.
